# Õppetund 10 - AI-agendid tootmises

Selles õppetunnis õpite **tootmismustreid** AI-agendidele, kasutades Microsoft Agent Frameworki (Python). Käsitleme:

- **Jälgitavus** — agentide suhtlustele ajastuse ja logimise lisamine
- **Hindamine** — hindamisagendi kasutamine vastuste kvaliteedi hindamiseks
- **Kulude juhtimine** — strateegiad tokenite optimeerimiseks ja mudeli valimiseks

Stsenaarium on **reisiplaneerija**, mis aitab kasutajatel reise planeerida, millele on lisatud jälgimine ja hindamine.


## Seadistamine


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Tootmiskaalutlused

Tehisintellekti agentide viimine prototüüpidest tootmisse nõuab hoolikat tähelepanu kolmele sambale:

1. **Jälgitavus** — Teil peab olema ülevaade sellest, mida agent teeb, kui kaua see aega võtab, ja milliseid tööriistu ta kutsub. Ilma jälgimise ja logimiseta on tootmiskeskkonna probleemide silumine peaaegu võimatu.

2. **Hindamine** — Automatiseeritud kvaliteedikontrollid tagavad, et agendi vastused jäävad ajas täpseteks, täielikeks ja kasulikeks. Hindamisagent võib vastuseid hinnata määratletud kriteeriumite alusel.

3. **Kulude juhtimine** — Tokenite kasutamine mõjutab otseselt kulusid. Strateegiad nagu päringu optimeerimine, mudeli valik ja vahemälu kasutamine aitavad kulusid kontrolli all hoida ilma kvaliteeti ohverdamata.


## Observeeritava agendi loomine

Määratleme reisi tööriistad ja ümbritseme agendi väljakutse ajastamisega, et saaksime jälgida latentsusaega. Tootmises integreeriksite OpenTelemetry või sarnase jälgimise tagaplaaniga.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Hindamismustrid

Tavaline tootmismuster on kasutada teist agenti kui **hindaja**. Hindaja hindab peamise agendi vastust eelmääratletud kriteeriumite alusel, nagu täielikkus, täpsus ja kasulikkus.

See võimaldab:
- Automatiseeritud kvaliteedikontrolle enne, kui vastused kasutajateni jõuavad
- Regressiooni tuvastamist siis, kui promptid või mudelid muutuvad
- Agendi jõudluse pidevat jälgimist aja jooksul


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kulu juhtimise strateegiad

Kulude kontrollimine on tootmisalastes AI-agentides ülioluline. Siin on peamised strateegiad:

| Strateegia | Kirjeldus |
|---|---|
| **Prompti optimeerimine** | Hoia süsteemi juhised lühikesed. Eemalda üleliigne kontekst, et vähendada sisendtokeneid. |
| **Mudeli valik** | Kasuta väiksemaid, odavamaid mudeleid (nt GPT-4o-mini) lihtsate ülesannete, nagu klassifitseerimine või eraldamine, jaoks ning jäta suuremad mudelid keerulisema arutelu jaoks. |
| **Vahemälu kasutamine** | Salvesta tööriistade tulemused ja sagedased päringud vahemällu, et vältida korduvaid API-kõnesid. |
| **Tokeni eelarved** | Sea `max_tokens` piirangud, et vältida ettearvamatult pikki vastuseid. |
| **Grupeerimine** | Grupeeri mitu kasutajapäringut üheks API-kõneks, kus võimalik. |

Praktikas toimib hästi kihiline lähenemine: suuna lihtsad päringud kiirele ja odavale mudelile ning eskaleeri ainult keerulised päringud võimsamale mudelile.


## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Lisada jälgitavust** agendi interaktsioonidele aegade mõõtmise ja logimise abil, luues aluse jälgimiseks ja monitooringuks.
2. **Hinnata agendi vastuseid** automaatselt, kasutades hindamisagenti, mis annab skoori täielikkuse, täpsuse ja kasulikkuse kohta.
3. **Hallata kulusid** läbi prompti optimeerimise, mudeli valiku, vahemällu salvestamise ja tokenite eelarvete.

Need tootmispraktikad aitavad tagada, et teie tehisintellekti agendid on skaalal usaldusväärsed, mõõdetavad ja kulutõhusad.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vastutusest loobumine:
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüdleme täpsuse poole, tuleb arvestada, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle algkeeles tuleks pidada autoriteetseks allikaks. Kriitilise teabe puhul soovitame kasutada professionaalset inimtõlget. Me ei vastuta mis tahes arusaamatuste ega väärtõlgenduste eest, mis võivad tuleneda selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
